# <a id='toc1_'></a>[Data preparation](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [Data preparation](#toc1_)    
    - [Config](#toc1_1_1_)    
    - [Import dependencies and load data](#toc1_1_2_)    
    - [Process raw data](#toc1_1_3_)    
      - [Convert date features to age in months](#toc1_1_3_1_)    
      - [Remove redundant raw value features of Z scaled values](#toc1_1_3_2_)    
      - [Remove less relevant or inaccessible features](#toc1_1_3_3_)    
    - [Renaming variables for clarity](#toc1_1_4_)    
    - [Feature completeness assessment and data export](#toc1_1_5_)    
      - [Renal function set for infants (First TUR <= 12 months)](#toc1_1_5_1_)    
      - [Renal function set of children (First TUR > 12 months)](#toc1_1_5_2_)    
      - [CKD5 positive set](#toc1_1_5_3_)    
      - [Bladder function set](#toc1_1_5_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

### <a id='toc1_1_1_'></a>[Config](#toc0_)

In [ ]:
# logging setup
from puv.logging_setup import setup
setup()  

import logging
logger = logging.getLogger(__name__)
logger.info("Logging initialized.")

In [ ]:
# Environment setup
from pathlib import Path

# Define the base directory for the project
BASE = Path('/workspaces/CODESPACE/').resolve()

# Define paths for data and analysis directories & ensure key folders exist
RAW = BASE / "data" / "raw" / "2025_puv_jaroy-lundar_raw-data.xlsx"
PREP = BASE / "data" / "prep"
FIGS = BASE / "results" / "eda"

for p in [PREP, FIGS]:
    p.mkdir(parents=True, exist_ok=True)

logger.info("Project base set to: %s", BASE)

# Constants and Configuration
THRESHOLD = 75 # Threshold value for analysis data completeness
MONTH_DAYS = 30.44 # Average number of days used to define a month

### <a id='toc1_1_2_'></a>[Import dependencies and load data](#toc0_)

In [ ]:
# Standard dependencies
import pandas as pd
import puv.prep as prep

# Read raw data
df_raw = pd.read_excel(RAW)

# Uncomment to check raw data
# display(df_raw)

### <a id='toc1_1_3_'></a>[Process raw data](#toc0_)

#### <a id='toc1_1_3_1_'></a>[Convert date features to age in months](#toc0_)

Convert values in all date columns to the number of months since the patient's date of birth. 

In [ ]:
# Create a copy of the raw data for date and time processing
df_prep = df_raw.copy()

# Explicitly convert the Date of birth collumn to datetime
df_prep['Date of birth'] = pd.to_datetime(df_prep['Date of birth'])

# Iterate over date columns (excluding 'Date of birth') and calculate time deltas in months
for column in df_prep.columns:
    if column != 'Date of birth' and pd.api.types.is_datetime64_any_dtype(df_prep[column]):
        timedelta_months = (df_prep[column] - df_prep['Date of birth']) / pd.Timedelta(days=MONTH_DAYS)
     
        # Replace original date values with months relative to 'Date of birth'
        df_prep[column] = timedelta_months.round(2)

# Rename columns from 'Date' to 'Age', and 'date' to 'age'. Exempt 'Date of birth'
df_prep_all = df_prep.rename(columns=lambda x: x.replace('Date', 'Age').replace('date', 'age') if x != 'Date of birth' else x)

# Save the prepared data to an Excel file
df_prep_all.to_excel(PREP / 'data_all.xlsx', index=False)

# Uncomment to check df_prep_ageinmonths
# display (df_prep_all)


#### <a id='toc1_1_3_2_'></a>[Remove redundant raw value features of Z scaled values](#toc0_)

Note that these Z-scale values are pre-calculated based on standard distributions observed in the general population

In [ ]:
# Create a copy of the DataFrame for trimming of redundant columns
df_trimmed = df_prep_all.copy()

# Remove columns that are not needed for the analysis as these are better represented by the z scaled values (relative to the general population)

redundant_columns = [   
'F Creatinine (first)',
'Baseline creatinine',
'Nadir creatinine',
'1 Creatinine (1 year)',
'2 Creatinine (2 years)',
'4 Creatinine (4 years)',
'6 Creatinine (6 years)',
'L Creatinine (last)',
'Birth weight (g)',
'Birth length (cm)',
'1 Weight (1 year, kg)',
'2 Weight (2 years, kg)',
'4 Weight (4 years, kg)',
'6 Weight (6 years, kg)',
'L Weight (last, kg)',
'1 Height (1 year, cm)',
'2 Height (2 years, cm)',
'4 Height (4 years, cm)',
'6 Height (6 years, cm)',
'L Height (last, cm)',
'2 BMI (2 years, kg/m²)',
'4 BMI (4 years, kg/m²)',
'6 BMI (6 years, kg/m²)',
'L BMI (last, kg/m²)',
]

# Additinally also drop the 'Date of birth' column, as it is not needed for the analysis
redundant_columns.append('Date of birth')

df_trimmed = df_prep_all.drop(columns=redundant_columns)

# Uncomment to check the trimmed DataFrame
display(df_trimmed)


#### <a id='toc1_1_3_3_'></a>[Remove less relevant or inaccessible features](#toc0_)
Exclude the following features:

- All date-related lab features.
- All features that indicate general illness or hospitalization in early childhood.
- All features associated with ages above 1 year 


In [ ]:
# Remove columns with clinically uninformative date and age associated variables
# NB KEEP -->  'TUR age (first)' as a way to filter out older children. This refers to the age at which the first transurethral resection (TUR) was performed

# List of date and age associated variables to remove
date_features_to_remove = [
    'F Age LAB (first)',
    '1 Age LAB (1 year)',
    '2 Age LAB (2 years)',
    '4 Age LAB (4 years)',
    '6 Age LAB (6 years)',
    'L Age LAB (last follow up)',
 
    '1 Age growth (1 year)',
    '2 Age growth 2 years',
    '4 Age growth 4 years',
    '6 Age growth (6 years)',
    'L Age growth (last follow up)',
    
    'Age at continence (years)',
    'Age at flow (years)',
    'Age VCUG',
    'Age first ultrasound',
    'Age last follow up',
    'Age last ultrasound',
    'Age uroflowmetry',
    'Age catheter',
    'Age gastrostomy placement',
    'Age first intervention (catheter or TUR)'
]


# Remove features indicative of sick children, as these are not as relevant (nor sufficiently objective) for the analysis
sick_child_features_to_remove = [
    'Presenting symptom',
    'Parenteral nutrition (PN)',
    'Duration PN (days)',
    'Nasogatric tube',
    'Duration tube (days)',
    'Length of stay neonatal (days)',
    'Length of stay <1 year (days)',
    'Length of stay total (days)',
    'Z Birth length',
    'Z Birth weight',
    'Gestational age (weeks)',
    'Gastrostomy',
    'No. of admissions',
    'No. of procedures under general anesthesia',
    'Recurrent urinary tract infection',
    'Rec. UTI > 1 year',

    'F Hemoglobin (first)',
    '1 Hemoglobin (1 year)',
    'Z Height 1',

    'Z Crea F',

    'F Vitamin D (first)',
    '1 Vitamin D (1 year)',
    'F Iron (first)',
    'F Ferritin (first)',
    '1 Iron (1 year)',
    '1 Ferritin (1 year)',
    '1 Albumin (1 year)',
    'F Albumin (first)',

     'Continence daytime',
    'CIC (Catheterisation)',
    'Maxflow (ml/s)',
    'Voiding volume (ml)',
    'Residual urine (ml)',
    'eGFR at last follow up',

]

# Remove features that are present only in children older than 1 year, as these are not relevant for this analysis
features_older_than_1_year = [
    '2 Albumin (2 years)',
    '4 Albumin (4 years)',
    '6 Albumin (6 years)',
    'L Albumin (last)',

    '2 Ferritin (2 years)',
    '4 Ferritin (4 years)',
    '6 Ferritin (6 years)',
    'L Ferritin (last)',

    '2 Hemoglobin (2 years)',
    '4 Hemoglobin (4 years)',
    '6 Hemoglobin (6 years)',
    'L Hemoglobin (last)',

    '2 Iron (2 years)',
    '4 Iron (4 years)',
    '6 Iron (6 years)',
    'L Iron (last)',

    '2 Vitamin D (2 years)',
    '4 Vitamin D (4 years)',
    '6 Vitamin D (6 years)',
    'L Vitamin D (last)',

    'No dysplasia last US',
    'No hydronephrosis last US',
    'Unilateral dysplasia last US',
    'Unilateral hydronephrosis last US',
    'Bilateral dysplasia last US',
    'Bilateral hydronephrosis last US',
    'Bladder wall last US',

    'Z BMI 2',
    'Z BMI 4',
    'Z BMI 6',
    'Z BMI L',

    'Z Crea 1',
    'Z Crea 2',
    'Z Crea 4',
    'Z Crea 6',
    'Z Crea L',

    'Z Height 2',
    'Z Height 4',
    'Z Height 6',
    'Z Height L',

    'Z Weight 2',
    'Z Weight 4',
    'Z Weight 6',
    'Z Weight L',
]

# Combine the three lists
cols_to_remove = (
    date_features_to_remove + 
    sick_child_features_to_remove + 
    features_older_than_1_year
)

# Remove columns in df_trimmed if they're in the cols_to_remove list
df_processed = df_trimmed.drop(columns=cols_to_remove).copy()

# Uncomment to check the processed DataFrame features
# display(sorted(df_processed.columns.to_list()))


### <a id='toc1_1_4_'></a>[Renaming variables for clarity](#toc0_)

In [ ]:
# Features to be renamed
old_column_names =[
    'Z Crea Baseline',
    'Z Weight 1',
    'Z Crea nadir',
    'Bladder wall first US',
    'No hydronephrosis first US',
    'No VUR',
    'Urinoma/Ascites first US',
    'Bilateral hydronephrosis first US',
    'Unilateral hydronephrosis first US',
    'Unilateral dysplasia first US',
    'No dysplasia first US',
    'VUR Unilateral',
    'Prenatal diagnosis',
    'Bilateral dysplasia first US',
    'VUR bilateral',
]

# New feature names
new_column_names = [
    'Baseline creatinine',
    'Weight (1 year)',
    'Nadir creatinine',
    'Thick bladder wall',
    'No hydronephrosis',
    'No VUR',
    'Urinoma and/or Acites',
    'Bilateral hydronephrosis',
    'Unilateral hydronephrosis',
    'Unilateral dysplasia',
    'No dysplasia',
    'Unilateral VUR',
    'Prenatal diagnosis',
    'Bilateral dysplasia',
    'Bilateral VUR',
]

# Check that all features to be renamed are in the dataframe
missing = [name for name in old_column_names if name not in df_processed.columns.to_list()]

if missing:
    logger.info(f"Missing columns: {missing}")
else:
    logger.info("All old columns are present.")


# Create a mapping dictionary and apply renaming
rename_map = dict(zip(old_column_names, new_column_names))
df_processed_rn = df_processed.rename(columns=rename_map)

# Save the prepared data to an Excel file
df_processed_rn.to_excel(PREP / 'data_processed.xlsx', index=False)

# Uncomment to display 
# display(df_processed_rn)


### <a id='toc1_1_5_'></a>[Feature completeness assessment and data export](#toc0_)
1. Renal set       – Includes patients and relevant features for renal function analysis.

2. Bladder set     – Includes patients and relevant features for those diagnosed with bladder dysfunction.

3. CKD5 set        – Includes patients who are positive for Chronic Kidney Disease (CKD) stage 5.

#### <a id='toc1_1_5_1_'></a>[Renal function set for infants (First TUR <= 12 months)](#toc0_)

In [ ]:
# Make a  copy of the df_processed_rn for the renal function set
renal_set = df_processed_rn.copy()

# Remove clinically uninformative features for exploring renal function
renal_set_remove = [
    'Bladder catheter',
    'Bladder dysfunction',
]

# Drop the clinically uninformative features
renal_set_filtered = renal_set.drop(columns=renal_set_remove)

# retain only patient data pertaining to infants -- i.e., less than or equal to 12 months old (based on the first TUR age)
renal_set_infants = renal_set_filtered[renal_set_filtered['TUR age (first)'] <= 12].reset_index(drop=True)


# Create a copy to hold and not subject to the completeness filter 
# Remove the non_diagnostic features
hold_not_diagnostic = ['Patient number', 'CKD5',  'Renal function at last follow up']
renal_hold = renal_set_infants[hold_not_diagnostic].copy()
renal_set_infants = renal_set_infants.drop(['TUR age (first)', 'CKD5', 'Renal function at last follow up'], axis=1) # keep the Patient number for reference

# # Filter and review completeness of features. Must have at least 75% of the data present
# Also remove the 'Albumin' and 'Hemoglobin' features, as these are not relevant for the renal function set
renal_infants_review = prep.filter_and_review_completeness(renal_set_infants, threshold=THRESHOLD, out_path= FIGS / 'Renal_set_infants_completeness', title='Renal set infants')

# Merge the renal set with the hold features on 'Patient number'
renal_infants_complete = pd.merge(renal_infants_review, renal_hold, on='Patient number', how='left')

# save the filtered renal set
renal_infants_complete.to_excel(PREP / 'renal_set_infants.xlsx', index=False)

logger.info(f'# of patients in the renal set: {renal_set.shape[0]}')
logger.info(f'# of infant patients in the filtered renal set: {renal_infants_complete.shape[0]}')

#### <a id='toc1_1_5_2_'></a>[Renal function set of children (First TUR > 12 months)](#toc0_)

In [ ]:
# retain only patient data pertaining to children -- i.e., greater than 12 months old (based on the first TUR age)
renal_set_children = renal_set_filtered[renal_set_filtered['TUR age (first)'] > 12].reset_index(drop=True) # renal_set_filtered is from before

# Create a copy  to hold and remove the non_diagnostic features, remove irrelevant columns
hold_not_diagnostic = ['Patient number', 'CKD5',  'Renal function at last follow up']
renal_hold = renal_set_children[hold_not_diagnostic].copy()
renal_set_children = renal_set_children.drop(['TUR age (first)', 'CKD5', 'Renal function at last follow up', 'Rec. UTI < 1 year'], axis=1)

# # Filter and review completeness of features. Must have at least 75% of the data present
# Also remove the 'Albumin' and 'Hemoglobin' features, as these are not relevant for the renal function set
renal_children_review = prep.filter_and_review_completeness(renal_set_children, threshold=THRESHOLD, out_path= FIGS / 'Renal_set_children_completeness', title='Renal set children', cols_to_remove = ['Albumin', 'Hemoglobin'])

# Merge the renal set with the hold features on 'Patient number'
renal_children_complete = pd.merge(renal_children_review, renal_hold, on='Patient number', how='left')

# save the filtered renal  set
renal_children_complete.to_excel(PREP / 'renal_set_children.xlsx', index=False)

logger.info(f'# of patients in the renal set: {renal_set.shape[0]}')
logger.info(f'# of child patients in the filtered renal set: {renal_children_complete.shape[0]}')

#### <a id='toc1_1_5_3_'></a>[CKD5 positive set](#toc0_)

In [ ]:
# retain only patient data pertaining to children with CKD5
CKD5_set = renal_set_filtered[renal_set_filtered['CKD5'] == 1].reset_index(drop=True) # renal_set_filtered is the filtered renal set from above

# Create a copy to hold  of non-diagnostic features and remove the diagnostic features
hold_not_diagnostic = ['Patient number', 'CKD5',  'Renal function at last follow up']
renal_hold = CKD5_set[hold_not_diagnostic].copy()
CKD5_set = CKD5_set.drop(['TUR age (first)', 'CKD5', 'Renal function at last follow up'], axis=1)

# # Filter and review completeness of features. Must have at least 75% of the data present
# Also remove the 'Albumin' and 'Hemoglobin' features, as these are not relevant for the renal function set
CKD5_set_review = prep.filter_and_review_completeness(CKD5_set, threshold=THRESHOLD, out_path = FIGS / "Renal_set_ckd5_completeness", title='Renal set CKD5', cols_to_remove = ['Albumin', 'Hemoglobin'])

# Merge the CKD set with the hold features on 'Patient number'
CKD5_set_complete = pd.merge(CKD5_set_review, renal_hold, on='Patient number', how='left')

# save the filtered CKD5 set
CKD5_set_complete.to_excel(PREP / 'renal_set_ckd5.xlsx', index=False)

logger.info(f'# of patients in the renal set: {renal_set.shape[0]}')
logger.info(f'# of CKD5 patients in the filtered renal set: {CKD5_set_complete.shape[0]}')

#### <a id='toc1_1_5_4_'></a>[Bladder function set](#toc0_)

In [ ]:
bladder_dysfunction_set = df_processed_rn.copy()

# Remove rows with missing values in the 'Bladder dysfunction' column
bladder_dysfunction_set = bladder_dysfunction_set[bladder_dysfunction_set['Bladder dysfunction'].notnull()].reset_index(drop=True)

# Create a copy to hold of non-diagnostic features and remove the diagnostic features
hold_not_diagnostic = ['Patient number', 'Bladder dysfunction', 'CKD5', 'Renal function at last follow up']
bladder_hold = bladder_dysfunction_set[hold_not_diagnostic].copy()
bladder_dysfunction_set = bladder_dysfunction_set.drop(['TUR age (first)', 'Bladder dysfunction', 'CKD5', 'Renal function at last follow up'], axis=1)

# filter and review missingness
bladder_dysfunction_set_review = prep.filter_and_review_completeness(bladder_dysfunction_set, threshold=THRESHOLD, out_path= FIGS / 'Bladder_set_completeness', title='Bladder set')

# Merge the Bladder set with the hold features on 'Patient number'
bladder_dysfunction_set_complete = pd.merge(bladder_dysfunction_set_review, bladder_hold, on='Patient number', how='left')

# save the filtered bladder function set
bladder_dysfunction_set_complete.to_excel(PREP / 'bladder_set.xlsx', index=False)

logger.info(f'# of patients in the bladder dysfunction set: {bladder_dysfunction_set.shape[0]}')
logger.info(f'# of patients in the filtered bladder dysfunction set: {bladder_dysfunction_set_complete.shape[0]}')
